# EMG Autoencoder — Vanilla v4 → `_latent_v3.hdf5` per session

Trains a **fresh autoencoder on each session individually**, then saves compressed latents.

- **Input**: all `89335547/*.hdf5` (raw 2000 Hz, non-latent, non-500hz)
- **All 32 channels**, 512-sample window (256 ms), stride 64, bandpass 60–500 Hz
- **20 epochs** per session, batch 256, Adam + cosine decay, latent dim 256
- **Output**: `89335547/*_latent_v3.hdf5` for every session + `ae_vanilla_v4_<session>.pt`

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'unidecode'], check=True)

CompletedProcess(args=['/Users/jonathangray/miniconda3/envs/e219/bin/python', '-m', 'pip', 'install', '-q', 'h5py', 'unidecode'], returncode=0)

In [2]:
import sys
sys.path.insert(0, '..')

import h5py
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

from emg2qwerty.charset import charset as get_charset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHARSET = get_charset()
print('device:', device)

device: cpu


In [ ]:
# ── Conformer CTC on reconstructed EMG — all sessions ────────────────────────
import h5py, json, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

SESSION_DIR  = Path('89335547')
recons_files = sorted(SESSION_DIR.glob('*_recons_v3.hdf5'))
print(f'Total sessions: {len(recons_files)}')

CHUNK   = 256
RATE_HZ = 2000 / 64

# ── Shared vocabulary ─────────────────────────────────────────────────────────
all_keys = set()
for f_path in recons_files:
    with h5py.File(f_path) as f:
        ks = json.loads(f['emg2qwerty'].attrs['keystrokes'])
        all_keys.update(k.get('key', '') for k in ks if k.get('key'))

blank_id    = 0
unique_keys = sorted(all_keys)
key_to_id   = {k: i + 1 for i, k in enumerate(unique_keys)}
id_to_key   = {v: k for k, v in key_to_id.items()}
num_classes = len(unique_keys) + 1
print(f'Vocabulary: {num_classes} classes  ({len(unique_keys)} keys + blank)')

# ── Session-level train / val / test split ────────────────────────────────────
n_test = max(1, round(len(recons_files) * 0.10))   # last ~10%  → test
n_val  = max(1, round(len(recons_files) * 0.15))   # next ~15%  → val
# remainder                                         # first ~75% → train

test_files  = recons_files[-(n_test):]
val_files   = recons_files[-(n_val + n_test):-n_test]
train_files = recons_files[:-(n_val + n_test)]

print(f'\nSplit (sessions):')
print(f'  train : {len(train_files)}')
print(f'  val   : {len(val_files)}')
print(f'  test  : {len(test_files)}')
print(f'\nTest sessions (held out until final eval):')
for f in test_files:
    print(f'  {f.name[:70]}')

# ── Load + chunk ──────────────────────────────────────────────────────────────
def load_chunks(file_list):
    X, tgt = [], []
    for f_path in file_list:
        with h5py.File(f_path) as f:
            times      = f['emg2qwerty/timeseries/time'][:]
            emg_left   = f['emg2qwerty/timeseries/emg_left'][:]
            emg_right  = f['emg2qwerty/timeseries/emg_right'][:]
            keystrokes = json.loads(f['emg2qwerty'].attrs['keystrokes'])
        emg = np.concatenate([emg_left, emg_right], axis=1).astype(np.float32)
        emg = (emg - emg.mean(0, keepdims=True)) / (emg.std(0, keepdims=True) + 1e-8)
        keys_sorted = sorted(keystrokes, key=lambda k: k['start'])
        for start in range(0, len(emg) - CHUNK + 1, CHUNK):
            end        = start + CHUNK
            t0_chunk   = times[start]
            t1_chunk   = times[end - 1] + 1.0 / RATE_HZ
            chunk_keys = [k for k in keys_sorted
                          if t0_chunk <= k['start'] < t1_chunk and k.get('key') in key_to_id]
            X.append(emg[start:end])
            tgt.append(torch.tensor([key_to_id[k['key']] for k in chunk_keys], dtype=torch.long))
    return torch.tensor(np.array(X), dtype=torch.float32), tgt

X_train, tgt_train = load_chunks(train_files)
X_val,   tgt_val   = load_chunks(val_files)
X_test,  tgt_test  = load_chunks(test_files)   # loaded but NOT used until final eval

print(f'\nChunks — train: {len(X_train)}  val: {len(X_val)}  test: {len(X_test)}')
print(f'Avg keys/chunk: {np.mean([len(t) for t in tgt_train + tgt_val]):.1f}')

# ── SpecAugment ───────────────────────────────────────────────────────────────
class SpecAugment(nn.Module):
    def __init__(self, T_masks=2, T_width=20, C_masks=1, C_width=8, noise_std=0.05):
        super().__init__()
        self.T_masks = T_masks; self.T_width = T_width
        self.C_masks = C_masks; self.C_width = C_width
        self.noise_std = noise_std

    def forward(self, x):
        if not self.training: return x
        B, T, C = x.shape
        x = x.clone()
        for _ in range(self.T_masks):
            w = torch.randint(1, self.T_width + 1, (1,)).item()
            t = torch.randint(0, max(1, T - w), (B,))
            for b in range(B):
                x[b, t[b]:t[b]+w, :] = 0.0
        for _ in range(self.C_masks):
            w = torch.randint(1, self.C_width + 1, (1,)).item()
            c = torch.randint(0, max(1, C - w), (1,)).item()
            x[:, :, c:c+w] = 0.0
        if self.noise_std > 0:
            x = x + torch.randn_like(x) * self.noise_std
        return x

# ── Conformer blocks ──────────────────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x): return self.drop(x + self.pe[:, :x.size(1)])


class FeedForward(nn.Module):
    def __init__(self, d_model, expansion=4, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model * expansion), nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * expansion, d_model), nn.Dropout(dropout),
        )
    def forward(self, x): return x + 0.5 * self.net(x)


class ConvolutionModule(nn.Module):
    def __init__(self, d_model, kernel_size=31, dropout=0.1):
        super().__init__()
        assert kernel_size % 2 == 1
        self.norm = nn.LayerNorm(d_model)
        self.pw1  = nn.Linear(d_model, d_model * 2)
        self.dw   = nn.Conv1d(d_model, d_model, kernel_size,
                              padding=kernel_size // 2, groups=d_model)
        self.ln   = nn.LayerNorm(d_model)
        self.act  = nn.SiLU()
        self.pw2  = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        r = x
        x = F.glu(self.pw1(self.norm(x)), dim=-1)
        x = self.act(self.ln(self.dw(x.transpose(1, 2)).transpose(1, 2)))
        return r + self.drop(self.pw2(x))


class ConformerBlock(nn.Module):
    def __init__(self, d_model, n_heads=4, kernel_size=31, dropout=0.1):
        super().__init__()
        self.ff1       = FeedForward(d_model, dropout=dropout)
        self.norm      = nn.LayerNorm(d_model)
        self.attn      = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.attn_drop = nn.Dropout(dropout)
        self.conv      = ConvolutionModule(d_model, kernel_size, dropout)
        self.ff2       = FeedForward(d_model, dropout=dropout)
        self.out_norm  = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.ff1(x)
        r = x; xn = self.norm(x)
        x = r + self.attn_drop(self.attn(xn, xn, xn, need_weights=False)[0])
        x = self.conv(x)
        x = self.ff2(x)
        return self.out_norm(x)


class ConformerCTC(nn.Module):
    def __init__(self, in_dim=32, d_model=128, n_heads=4, n_layers=4,
                 kernel_size=31, dropout=0.1, num_classes=num_classes):
        super().__init__()
        self.proj   = nn.Linear(in_dim, d_model)
        self.pe     = PositionalEncoding(d_model, max_len=CHUNK, dropout=dropout)
        self.blocks = nn.ModuleList([
            ConformerBlock(d_model, n_heads, kernel_size, dropout)
            for _ in range(n_layers)
        ])
        self.fc = nn.Linear(d_model, num_classes)
        nn.init.constant_(self.fc.bias[blank_id], 1.0)

    def forward(self, x):
        x = self.pe(self.proj(x))
        for block in self.blocks:
            x = block(x)
        return self.fc(x)


augment = SpecAugment(T_masks=2, T_width=20, C_masks=1, C_width=8, noise_std=0.05)
model   = ConformerCTC(in_dim=32, d_model=128, n_heads=4, n_layers=4,
                       kernel_size=31, dropout=0.3).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Params: {n_params:,}')

ctc_loss = nn.CTCLoss(blank=blank_id, zero_infinity=True)
opt      = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

NUM_EPOCHS  = 120
BATCH_SIZE  = 32
steps_total = NUM_EPOCHS * max(1, len(X_train) // BATCH_SIZE)
steps_warm  = steps_total // 10
input_lens  = torch.full((BATCH_SIZE,), CHUNK, dtype=torch.long)
step        = 0

def lr_lambda(s):
    if s < steps_warm: return s / max(1, steps_warm)
    p = (s - steps_warm) / max(1, steps_total - steps_warm)
    return 0.5 * (1 + math.cos(math.pi * p))

sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

# ── Training loop (train + val only — test never touched here) ────────────────
def run_epoch(X_split, tgt_split, train=True):
    global step
    total_loss, blank_sum, n = 0.0, 0.0, 0
    idx = np.random.permutation(len(X_split)) if train else np.arange(len(X_split))
    for i in range(0, len(X_split) - BATCH_SIZE + 1, BATCH_SIZE):
        bi       = idx[i:i + BATCH_SIZE]
        xb       = X_split[bi].to(device)
        if train: xb = augment(xb)
        tgt      = torch.cat([tgt_split[j] for j in bi])
        tgt_lens = torch.tensor([len(tgt_split[j]) for j in bi], dtype=torch.long)
        logits   = model(xb)
        log_prob = F.log_softmax(logits, dim=-1).permute(1, 0, 2)
        loss     = ctc_loss(log_prob, tgt, input_lens[:BATCH_SIZE], tgt_lens)
        if train:
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); step += 1
        blank_sum  += (logits.argmax(-1) == blank_id).float().mean().item()
        total_loss += loss.item(); n += 1
    return total_loss / max(n, 1), blank_sum / max(n, 1)

train_losses, val_losses = [], []
best_val_loss, best_state = float('inf'), None

print(f'\n{"epoch":>5}  {"train":>8}  {"val":>8}  {"blank%":>7}  {"best":>8}')
for epoch in range(NUM_EPOCHS):
    model.train(); augment.train()
    tl, _ = run_epoch(X_train, tgt_train, train=True)
    model.eval(); augment.eval()
    with torch.no_grad(): vl, bf = run_epoch(X_val, tgt_val, train=False)
    train_losses.append(tl); val_losses.append(vl)
    if vl < best_val_loss:
        best_val_loss = vl
        best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'{epoch+1:5d}  {tl:8.4f}  {vl:8.4f}  {bf*100:6.1f}%  {best_val_loss:8.4f}')

model.load_state_dict(best_state)
print(f'\nRestored best checkpoint  val={best_val_loss:.4f}')

# ── Greedy decode ─────────────────────────────────────────────────────────────
def greedy_decode(log_probs):
    tokens = log_probs.argmax(-1).tolist()
    out, prev = [], None
    for t in tokens:
        if t != blank_id and t != prev:
            out.append(id_to_key.get(t, '?'))
        prev = t
    return out

def eval_cer(X_split, tgt_split, label=''):
    def edit_distance(ref, hyp):
        R, H = len(ref), len(hyp)
        dp = np.zeros((R + 1, H + 1), dtype=int)
        dp[:, 0] = np.arange(R + 1); dp[0, :] = np.arange(H + 1)
        for r in range(1, R + 1):
            for h in range(1, H + 1):
                dp[r, h] = dp[r-1, h-1] if ref[r-1] == hyp[h-1] else \
                           1 + min(dp[r-1, h], dp[r, h-1], dp[r-1, h-1])
        return dp[R, H]

    total_dist, total_chars = 0, 0
    with torch.no_grad():
        for i in range(len(X_split)):
            lp   = F.log_softmax(model(X_split[i:i+1].to(device)), dim=-1)[0].cpu()
            pred = greedy_decode(lp)
            true = [id_to_key[t.item()] for t in tgt_split[i]]
            total_dist  += edit_distance(true, pred)
            total_chars += max(len(true), 1)
    cer = total_dist / total_chars
    print(f'{label:8s}  CER: {cer:.3f}  ({cer*100:.1f}%)  '
          f'[{total_dist} edits / {total_chars} chars]')
    return cer

model.eval()
print('\n── Final evaluation ──')
eval_cer(X_val,  tgt_val,  label='val')
eval_cer(X_test, tgt_test, label='test')   # ← first time test is touched

# ── Sample predictions (val) ──────────────────────────────────────────────────
print('\n── Val samples ──')
with torch.no_grad():
    for i in range(min(5, len(X_val))):
        lp   = F.log_softmax(model(X_val[i:i+1].to(device)), dim=-1)[0].cpu()
        pred = greedy_decode(lp)
        true = [id_to_key[t.item()] for t in tgt_val[i]]
        print(f'  true={true}')
        print(f'  pred={pred}\n')

# ── Loss curve ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(train_losses, label='train')
ax.plot(val_losses,   label='val')
ax.set_xlabel('Epoch'); ax.set_ylabel('CTC loss')
ax.set_title('Conformer CTC — reconstructed EMG')
ax.legend(); ax.grid(True, lw=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── Character Error Rate (CER) ────────────────────────────────────────────────
def edit_distance(ref, hyp):
    """Levenshtein distance between two lists."""
    R, H = len(ref), len(hyp)
    dp = np.zeros((R + 1, H + 1), dtype=int)
    dp[:, 0] = np.arange(R + 1)
    dp[0, :] = np.arange(H + 1)
    for r in range(1, R + 1):
        for h in range(1, H + 1):
            if ref[r-1] == hyp[h-1]:
                dp[r, h] = dp[r-1, h-1]
            else:
                dp[r, h] = 1 + min(dp[r-1, h],    # deletion
                                   dp[r, h-1],    # insertion
                                   dp[r-1, h-1])  # substitution
    return dp[R, H]

model.eval()
total_dist, total_chars = 0, 0

with torch.no_grad():
    for i in range(len(X_val)):
        lp   = F.log_softmax(model(X_val[i:i+1].to(device)), dim=-1)[0].cpu()
        pred = greedy_decode(lp)
        true = [id_to_key[t.item()] for t in tgt_val[i]]

        total_dist  += edit_distance(true, pred)
        total_chars += max(len(true), 1)

cer = total_dist / total_chars
print(f'CER: {cer:.3f}  ({cer*100:.1f}%)')
print(f'(total chars: {total_chars}  total edits: {total_dist})')